In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import the CRUD Python module created in Project One
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################

username = os.environ["AAC_MONGO_USERNAME"]
password = os.environ["AAC_MONGO_PASSWORD"]

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Sending the read method an empty document returns all documents
df = pd.DataFrame.from_records(db.read({}))

# Remove MongoDB ObjectId because Dash cannot display it directly
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

# Replace missing values with None for the dashboard
df = df.astype(object).where(pd.notna(df), None)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Add Grazioso Salvare's logo
image_filename = 'Grazioso Salvare Logo.png'

with open(image_filename, 'rb') as image_file:
    encoded_image = base64.b64encode(image_file.read())

app.layout = html.Div([
    # Required logo linked to SNHU and unique identifier
    html.Center([
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(
                    encoded_image.decode()
                ),
                alt='Grazioso Salvare logo',
                style={
                    'width': '180px',
                    'maxWidth': '100%'
                }
            ),
            href='https://www.snhu.edu',
            target='_blank'
        ),
        html.B(html.H1('CS-340 Dashboard')),
        html.H3('Developed by Alberto Rodriguez')
    ]),

    html.Hr(),

    # Interactive rescue-type filtering options
    html.Div(
        dcc.RadioItems(
            id='filter-type',
            options=[
                {
                    'label': 'Water Rescue',
                    'value': 'water'
                },
                {
                    'label': 'Mountain or Wilderness Rescue',
                    'value': 'mountain'
                },
                {
                    'label': 'Disaster or Individual Tracking',
                    'value': 'disaster'
                },
                {
                    'label': 'Reset',
                    'value': 'reset'
                }
            ],
            value='reset',
            labelStyle={
                'display': 'inline-block',
                'marginRight': '20px',
                'padding': '8px'
            }
        )
    ),

    html.Hr(),

    # Interactive data table
    dash_table.DataTable(
        id='datatable-id',

        columns=[
            {
                'name': i,
                'id': i,
                'deletable': False,
                'selectable': True
            }
            for i in df.columns
        ],

        data=df.to_dict('records'),

        editable=False,
        filter_action='native',
        sort_action='native',
        sort_mode='multi',

        column_selectable='single',
        selected_columns=[],

        row_selectable='single',
        row_deletable=False,
        selected_rows=[0] if not df.empty else [],

        page_action='native',
        page_current=0,
        page_size=10,

        style_table={
            'overflowX': 'auto'
        },

        style_cell={
            'textAlign': 'left',
            'minWidth': '100px',
            'width': '150px',
            'maxWidth': '180px',
            'whiteSpace': 'normal'
        },

        style_header={
            'fontWeight': 'bold'
        }
    ),

    html.Br(),
    html.Hr(),

    # Display the chart and map side by side.
    # Allow them to wrap on smaller screens.
    html.Div(
        className='row',
        style={
            'display': 'flex',
            'flexWrap': 'wrap',
            'gap': '20px'
        },
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={
                    'flex': '1 1 400px',
                    'minWidth': '0'
                }
            ),

            html.Div(
                id='map-id',
                className='col s12 m6',
                style={
                    'flex': '1 1 400px',
                    'minWidth': '0'
                }
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    [
        Output('datatable-id', 'data'),
        Output('datatable-id', 'selected_rows'),
        Output('datatable-id', 'page_current'),
        Output('datatable-id', 'filter_query'),
        Output('datatable-id', 'sort_by'),
        Output('datatable-id', 'selected_columns')
    ],
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    # Use the breed, sex, and age requirements from the specifications.
    # Age boundaries are inclusive.

    if filter_type == 'water':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$in': [
                    'Labrador Retriever Mix',
                    'Chesapeake Bay Retriever',
                    'Newfoundland'
                ]
            },
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {
                '$gte': 26,
                '$lte': 156
            }
        }

    elif filter_type == 'mountain':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$in': [
                    'German Shepherd',
                    'Alaskan Malamute',
                    'Old English Sheepdog',
                    'Siberian Husky',
                    'Rottweiler'
                ]
            },
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {
                '$gte': 26,
                '$lte': 156
            }
        }

    elif filter_type == 'disaster':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$in': [
                    'Doberman Pinscher',
                    'German Shepherd',
                    'Golden Retriever',
                    'Bloodhound',
                    'Rottweiler'
                ]
            },
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {
                '$gte': 20,
                '$lte': 300
            }
        }

    else:
        # Reset retrieves all animals
        query = {}

    # Retrieve matching documents through the Project One CRUD module
    dff = pd.DataFrame.from_records(db.read(query))

    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    dff = dff.astype(object).where(pd.notna(dff), None)

    data = dff.to_dict('records')
    selected_rows = [0] if data else []

    # Clear old table searches, sorting, and column selection.
    # Return to the first page when the rescue category changes.
    return data, selected_rows, 0, '', [], []


# Display breeds based on quantities represented in the filtered table
@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')],
    [State('datatable-id', 'data')]
)
def update_graphs(viewData, tableData):
    # Use current table data while derived_virtual_data initializes
    if viewData is None:
        viewData = tableData

    dff = pd.DataFrame.from_records(viewData or [])

    if dff.empty:
        return [
            html.H3('Breed Distribution'),
            html.P('No animals match the current filters.')
        ]

    breed_counts = (
        dff['breed']
        .fillna('Unknown')
        .value_counts()
        .rename_axis('breed')
        .reset_index(name='count')
    )

    figure = px.pie(
        breed_counts,
        names='breed',
        values='count',
        title='Breed Distribution of Filtered Animals'
    )

    # Keep labels inside slices to prevent overlapping outside labels.
    # Hovering still displays each breed and its animal count.
    figure.update_traces(
        textposition='inside',
        textinfo='percent',
        hovertemplate='%{label}: %{value} animals<extra></extra>'
    )

    # Hide labels that cannot fit while retaining every breed.
    figure.update_layout(
        height=500,
        uniformtext_minsize=12,
        uniformtext_mode='hide',
        margin=dict(t=60, b=20, l=20, r=20)
    )

    return [
        dcc.Graph(
            figure=figure,
            config={'responsive': True}
        )
    ]


# Highlight the selected table column
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [
        {
            'if': {'column_id': i},
            'backgroundColor': '#D2F3FF'
        }
        for i in (selected_columns or [])
    ]


# Update the map for the selected table entry.
# derived_virtual_data contains rows after table sorting/filtering.
# derived_virtual_selected_rows gives selection positions in those rows.
@app.callback(
    Output('map-id', 'children'),
    [
        Input('datatable-id', 'derived_virtual_data'),
        Input('datatable-id', 'derived_virtual_selected_rows')
    ],
    [State('datatable-id', 'data')]
)
def update_map(viewData, index, tableData):
    if viewData is None:
        viewData = tableData

    dff = pd.DataFrame.from_records(viewData or [])

    if dff.empty:
        return [
            html.H3('Animal Location'),
            html.P('No animals match the current filters.')
        ]

    # Default to the first matching animal when no row is selected
    row = index[0] if index else 0

    if row < 0 or row >= len(dff):
        row = 0

    animal = dff.iloc[row]

    breed = animal.get('breed') or 'Unknown'
    animal_name = animal.get('name') or 'Unknown'

    # Access coordinates by field name rather than column position
    try:
        latitude = float(animal.get('location_lat'))
        longitude = float(animal.get('location_long'))

        valid_coordinates = (
            np.isfinite(latitude)
            and np.isfinite(longitude)
            and -90 <= latitude <= 90
            and -180 <= longitude <= 180
        )

    except (TypeError, ValueError):
        valid_coordinates = False

    # Show an Austin base map when coordinates are missing or invalid
    if not valid_coordinates:
        return [
            html.H3('Animal Location'),

            html.P(
                '{}: location unavailable. Showing Austin, Texas.'.format(
                    animal_name
                )
            ),

            dl.Map(
                style={
                    'width': '100%',
                    'height': '500px'
                },
                center=[30.2672, -97.7431],
                zoom=10,
                children=[
                    dl.TileLayer(id='base-layer-id')
                ]
            )
        ]

    return [
        html.H3('Animal Location'),

        dl.Map(
            style={
                'width': '100%',
                'height': '500px'
            },
            center=[latitude, longitude],
            zoom=10,
            children=[
                dl.TileLayer(id='base-layer-id'),

                dl.Marker(
                    position=[latitude, longitude],
                    children=[
                        dl.Tooltip(str(breed)),

                        dl.Popup([
                            html.H1('Animal Name'),
                            html.P(str(animal_name))
                        ])
                    ]
                )
            ]
        )
    ]


# Run app and display result.
# If port 8050 is already in use, stop the previous app
# or specify an alternate port, such as port=8051.
app.run_server()